# Notebook Overview — Normalize and Prepare Inputs

## Purpose

This notebook normalizes the **Digital Image Processing (DIP) feature vectors** and prepares final **machine learning–ready inputs** for both the **training** and **test** datasets.

It applies standard feature scaling using statistics derived from the training data, ensuring consistent and unbiased input representation for downstream model training and evaluation.

---

## Inputs

Required inputs:

* Feature vector CSV files:
  * `metadata/vectors/<train_feature_vectors_filename>`
  * `metadata/vectors/<test_feature_vectors_filename>`

Shared configuration:

* `src/project_config.py`

---

## Execution Model

* Training and test feature vector datasets are loaded
* Metadata and feature columns are separated
* Feature structure and dimensionality are validated
* A `StandardScaler` is fit using **training features only**
* The fitted scaler is applied to both training and test datasets
* Normalized feature matrices are recombined with metadata
* Final normalized datasets are validated and saved
* The fitted scaler is persisted for reuse
* Processing is deterministic and reproducible

---

## Outputs

**Normalized Training Feature Vectors**  
`metadata/vectors/<train_feature_vectors_normalized_filename>`

**Normalized Test Feature Vectors**  
`metadata/vectors/<test_feature_vectors_normalized_filename>`

**Saved Scaler**  
`metadata/models/scaler.pkl`

Each normalized dataset contains:

* Metadata columns:
  * `filename`
  * `class_label`
  * `source_dataset`
  * `subset`

* Feature columns (26 total), normalized to:

  * Mean ≈ 0  
  * Standard deviation ≈ 1  

---

## Expected Behavior

* Training features are standardized with zero mean and unit variance
* Test features are transformed using training-derived statistics
* Feature dimensionality remains consistent (26 features)
* Metadata remains unchanged and aligned with feature rows
* No missing values are present in normalized outputs
* Output datasets are ready for classifier training and evaluation
* Scaler can be reused for consistent preprocessing of new data

---
---

### 🔷 Step 1 — Startup: Environment and Verification

- Clone the GitHub repository if needed
- Add the project `src` directory to the Python path
- Import shared configuration values from `project_config.py`
- Define input paths for training and test feature vector CSV files
- Define output paths for normalized feature vectors
- Define paths for scaler storage and model metadata
- Create required output and model directories if they do not exist
- Verify that required feature vector input files are present
- Optionally display key configuration paths when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 1: Startup — Environment and Verification
# ============================================

# -------------------------------------------------
# Import required libraries
# -------------------------------------------------
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import StandardScaler

# -------------------------------------------------
# Notebook display control
# -------------------------------------------------
VERBOSE = True   # Set to False to reduce detailed output

# -------------------------------------------------
# Clone GitHub repository if needed
# -------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# -------------------------------------------------
# Add src directory to Python path
# -------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# -------------------------------------------------
# Import shared project configuration
# -------------------------------------------------
from project_config import (
    TRAIN_SUBSET,
    TEST_SUBSET,
    RANDOM_SEED,
    NUM_FEATURES,
    METADATA_COLUMNS,
    TRAIN_FEATURE_VECTOR_PATH,
    TEST_FEATURE_VECTOR_PATH,
    TRAIN_NORMALIZED_PATH,
    TEST_NORMALIZED_PATH,
    MODELS_METADATA_DIR,
    SCALER_PATH,
)

# -------------------------------------------------
# Convert configured paths to Path objects
# -------------------------------------------------
TRAIN_INPUT_FILE = Path(TRAIN_FEATURE_VECTOR_PATH)
TEST_INPUT_FILE = Path(TEST_FEATURE_VECTOR_PATH)

TRAIN_OUTPUT_FILE = Path(TRAIN_NORMALIZED_PATH)
TEST_OUTPUT_FILE = Path(TEST_NORMALIZED_PATH)

MODELS_DIR = Path(MODELS_METADATA_DIR)
SCALER_FILE = Path(SCALER_PATH)

OUTPUT_DIR = TRAIN_OUTPUT_FILE.parent

# -------------------------------------------------
# Ensure required output directories exist
# -------------------------------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Verify required input files
# -------------------------------------------------
print("Verifying required feature-vector CSV files...\n")

required_files = [
    TRAIN_INPUT_FILE,
    TEST_INPUT_FILE,
]

missing_files = [str(file_path) for file_path in required_files if not file_path.exists()]

if missing_files:
    raise FileNotFoundError(
        f"Missing required feature-vector files:\n{missing_files}"
    )

print("All required feature-vector CSV files are present.")

# -------------------------------------------------
# Optional verbose display of configuration paths
# -------------------------------------------------
if VERBOSE:
    print(f"Train input:      {TRAIN_INPUT_FILE}")
    print(f"Test input:       {TEST_INPUT_FILE}")
    print(f"Train output:     {TRAIN_OUTPUT_FILE}")
    print(f"Test output:      {TEST_OUTPUT_FILE}")
    print(f"Scaler file:      {SCALER_FILE}")
    print(f"Models directory: {MODELS_DIR}")

print("\nStartup complete.")



### 🔷 Step 2 — Display Input and Output Configuration

- Display input feature vector file paths for training and test datasets
- Display output file paths for normalized datasets
- Display path for saved scaler object
- Confirm configuration setup for normalization process

---

In [ ]:
# ============================================
# Step 2: Display Input and Output Configuration
# ============================================

# -------------------------------------------------
# Display input feature vector file paths
# -------------------------------------------------
print("Input files:")
print(f" - {TRAIN_INPUT_FILE}")
print(f" - {TEST_INPUT_FILE}")

# -------------------------------------------------
# Display output file paths
# -------------------------------------------------
print("\nOutput files:")
print(f" - {TRAIN_OUTPUT_FILE}")
print(f" - {TEST_OUTPUT_FILE}")
print(f" - {SCALER_FILE}")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nConfiguration display complete.")



### 🔷 Step 3 — Load Train and Test Feature Vectors

- Load training feature vector CSV file
- Load test feature vector CSV file
- Display shapes of loaded datasets for validation

---

In [ ]:
# ============================================
# Step 3: Load Train and Test Feature Vectors
# ============================================

# -------------------------------------------------
# Load training feature vector CSV
# -------------------------------------------------
df_train = pd.read_csv(TRAIN_INPUT_FILE)

# -------------------------------------------------
# Load test feature vector CSV
# -------------------------------------------------
df_test = pd.read_csv(TEST_INPUT_FILE)

# -------------------------------------------------
# Display summary of loaded datasets
# -------------------------------------------------
print("Train and test feature vectors loaded.\n")
print("df_train shape:", df_train.shape)
print("df_test shape: ", df_test.shape)



### 🔷 Step 4 — Validate Row Counts and Structure

- Identify feature columns by excluding metadata columns
- Verify that training and test datasets have identical column structure
- Display shapes of training and test datasets
- Validate number of metadata and feature columns
- Confirm feature count matches expected DIP feature dimension

---

In [ ]:
# ============================================
# Step 4: Validate Row Counts and Structure
# ============================================

# -------------------------------------------------
# Identify feature columns (exclude metadata)
# -------------------------------------------------
feature_columns = [col for col in df_train.columns if col not in METADATA_COLUMNS]

# -------------------------------------------------
# Validate column structure consistency
# -------------------------------------------------
assert list(df_train.columns) == list(df_test.columns), \
    "Train and test column structures do not match"

print("Column structure validated.")

# -------------------------------------------------
# Display dataset shapes
# -------------------------------------------------
print(f"\nTrain shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")

# -------------------------------------------------
# Validate feature column count
# -------------------------------------------------
print(f"\nNumber of metadata columns: {len(METADATA_COLUMNS)}")
print(f"Number of feature columns:  {len(feature_columns)}")

assert len(feature_columns) == NUM_FEATURES, \
    f"Expected {NUM_FEATURES} features, found {len(feature_columns)}"

print(f"\nFeature count validated ({NUM_FEATURES} DIP features).")



### 🔷 Step 5 — Separate Metadata and Feature Columns

- Validate expected number of feature columns
- Separate metadata columns from feature vector data
- Extract numeric feature matrices for training and test datasets
- Display shapes of metadata and feature matrices
- Optionally display sample metadata and feature rows when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 5: Separate Metadata and Feature Columns
# ============================================

# -------------------------------------------------
# Validate feature column count
# -------------------------------------------------
assert len(feature_columns) == NUM_FEATURES, \
    f"Expected {NUM_FEATURES} features, found {len(feature_columns)}"

# -------------------------------------------------
# Separate metadata columns
# -------------------------------------------------
train_meta = df_train[METADATA_COLUMNS].copy()
test_meta = df_test[METADATA_COLUMNS].copy()

# -------------------------------------------------
# Separate feature matrices (numeric inputs for ML)
# -------------------------------------------------
X_train = df_train[feature_columns].copy()
X_test = df_test[feature_columns].copy()

# -------------------------------------------------
# Display summary of separated data
# -------------------------------------------------
print("Metadata and feature matrices created.\n")

print("train_meta shape:", train_meta.shape)
print("test_meta shape: ", test_meta.shape)
print("X_train shape:   ", X_train.shape)
print("X_test shape:    ", X_test.shape)

# -------------------------------------------------
# Optional verbose display of sample data
# -------------------------------------------------
if VERBOSE:
    print("\nFirst 5 metadata rows (train):")
    display(train_meta.head())

    print("\nFirst 5 feature rows (train):")
    display(X_train.head())



### 🔷 Step 6 — Fit Scaler on Training Features

- Initialize `StandardScaler`
- Fit scaler using **training feature matrix only**
- Learn mean and standard deviation for each feature
- Confirm number of features learned by the scaler
- Optionally display sample feature means and standard deviations when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 6: Fit Scaler on Training Features
# ============================================

# -------------------------------------------------
# Initialize StandardScaler
# -------------------------------------------------
scaler = StandardScaler()

# -------------------------------------------------
# Fit scaler using training features only
# -------------------------------------------------
scaler.fit(X_train)

print("Scaler fitted on training feature matrix only.")
print("This scaler will be applied to both train and test data.\n")

# -------------------------------------------------
# Display learned scaler parameters
# -------------------------------------------------
print("Number of features learned by scaler:", len(scaler.mean_))

# -------------------------------------------------
# Optional verbose display of learned statistics
# -------------------------------------------------
if VERBOSE:
    print("\nFirst 5 feature means learned from training data:")
    for col, val in zip(feature_columns[:5], scaler.mean_[:5]):
        print(f"  {col}: {val:.6f}")

    print("\nFirst 5 feature standard deviations learned from training data:")
    for col, val in zip(feature_columns[:5], scaler.scale_[:5]):
        print(f"  {col}: {val:.6f}")



### 🔷 Step 7 — Transform Train and Test Features

- Apply fitted scaler to training and test feature matrices
- Normalize features using training-derived statistics
- Convert scaled arrays back to DataFrames
- Preserve column names and index alignment
- Display shapes of normalized datasets
- Optionally display sample normalized feature rows when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 7: Transform Train and Test Features
# ============================================

# -------------------------------------------------
# Apply fitted scaler to training and test features
# -------------------------------------------------
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------------------------------
# Convert scaled arrays back to DataFrames
# (preserve column names and index alignment)
# -------------------------------------------------
df_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=feature_columns,
    index=X_train.index
)

df_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=feature_columns,
    index=X_test.index
)

# -------------------------------------------------
# Display summary of normalized data
# -------------------------------------------------
print("Training and test feature matrices normalized.\n")

print("df_train_scaled shape:", df_train_scaled.shape)
print("df_test_scaled shape: ", df_test_scaled.shape)

# -------------------------------------------------
# Optional verbose display of normalized values
# -------------------------------------------------
if VERBOSE:
    print("\nFirst 5 normalized feature rows (train):")
    display(df_train_scaled.head())



### 🔷 Step 8 — Recombine Metadata with Normalized Features

- Reset indices to ensure proper alignment between metadata and feature matrices
- Concatenate metadata columns with normalized feature columns
- Construct final normalized datasets for training and test subsets
- Display shapes of recombined datasets
- Optionally display sample rows when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 8: Recombine Metadata with Normalized Features
# ============================================

# -------------------------------------------------
# Reset indices to ensure proper alignment
# -------------------------------------------------
train_meta = train_meta.reset_index(drop=True)
test_meta = test_meta.reset_index(drop=True)
df_train_scaled = df_train_scaled.reset_index(drop=True)
df_test_scaled = df_test_scaled.reset_index(drop=True)

# -------------------------------------------------
# Recombine metadata with normalized feature columns
# -------------------------------------------------
df_train_normalized = pd.concat([train_meta, df_train_scaled], axis=1)
df_test_normalized = pd.concat([test_meta, df_test_scaled], axis=1)

# -------------------------------------------------
# Display summary of recombined datasets
# -------------------------------------------------
print("Metadata recombined with normalized feature columns.\n")

print("df_train_normalized shape:", df_train_normalized.shape)
print("df_test_normalized shape: ", df_test_normalized.shape)

# -------------------------------------------------
# Optional verbose display of sample rows
# -------------------------------------------------
if VERBOSE:
    print("\nFirst 5 rows of normalized training table:")
    display(df_train_normalized.head())



### 🔷 Step 9 — Save Normalized Feature Vectors and Scaler

- Save normalized training and test feature vector datasets to CSV files
- Persist the fitted `StandardScaler` object for reuse
- Verify that all output files were successfully created
- Display output file locations for confirmation

---


In [ ]:
# ============================================
# Step 9: Save Normalized Feature Vectors and Scaler
# ============================================

# -------------------------------------------------
# Save normalized feature vector CSV files
# -------------------------------------------------
df_train_normalized.to_csv(TRAIN_OUTPUT_FILE, index=False)
df_test_normalized.to_csv(TEST_OUTPUT_FILE, index=False)

# -------------------------------------------------
# Persist trained scaler to disk
# -------------------------------------------------
joblib.dump(scaler, SCALER_FILE)

# -------------------------------------------------
# Verify output files were created
# -------------------------------------------------
if not TRAIN_OUTPUT_FILE.exists():
    raise FileNotFoundError(f"Train normalized file not created: {TRAIN_OUTPUT_FILE}")

if not TEST_OUTPUT_FILE.exists():
    raise FileNotFoundError(f"Test normalized file not created: {TEST_OUTPUT_FILE}")

if not SCALER_FILE.exists():
    raise FileNotFoundError(f"Scaler file not created: {SCALER_FILE}")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("Saved normalized feature vectors and scaler.\n")

print(f"Train normalized CSV: {TRAIN_OUTPUT_FILE}")
print(f"Test normalized CSV:  {TEST_OUTPUT_FILE}")
print(f"Scaler saved to:      {SCALER_FILE}")



### 🔷 Step 10 — Sanity Check Normalized Outputs

- Identify normalized feature columns (exclude metadata)
- Compute mean and standard deviation for training and test features
- Verify that training features are centered near 0 with unit variance
- Display compact deviation metrics for normalization validation
- Generate comparison table of train vs test statistics
- Optionally display detailed summary when `VERBOSE=True`
- Confirm normalization process is functioning correctly

---

In [ ]:
# ============================================
# Step 10: Sanity Check Normalized Outputs
# ============================================

# -------------------------------------------------
# Identify normalized feature columns
# -------------------------------------------------
normalized_feature_columns = [
    col for col in df_train_normalized.columns if col not in METADATA_COLUMNS
]

# -------------------------------------------------
# Compute summary statistics for train and test sets
# -------------------------------------------------
train_feature_means = df_train_normalized[normalized_feature_columns].mean()
train_feature_stds = df_train_normalized[normalized_feature_columns].std()

test_feature_means = df_test_normalized[normalized_feature_columns].mean()
test_feature_stds = df_test_normalized[normalized_feature_columns].std()

# -------------------------------------------------
# Display dataset shapes
# -------------------------------------------------
print("Sanity check on normalized outputs:\n")

print("Train normalized shape:", df_train_normalized.shape)
print("Test normalized shape: ", df_test_normalized.shape)

# -------------------------------------------------
# Compact normalization validation (training set)
# -------------------------------------------------
mean_abs = train_feature_means.abs().mean()
std_dev = (train_feature_stds - 1).abs().mean()

print(f"\nTrain mean deviation from 0 (avg abs): {mean_abs:.6f}")
print(f"Train std deviation from 1 (avg abs):  {std_dev:.6f}")

if mean_abs > 1e-3 or std_dev > 1e-3:
    print("\nWARNING: Normalization may be off (training values not near 0/1)")

# -------------------------------------------------
# Combined summary comparison table (train vs test)
# -------------------------------------------------
summary_df = pd.DataFrame({
    "train_mean": train_feature_means,
    "train_std": train_feature_stds,
    "test_mean": test_feature_means,
    "test_std": test_feature_stds,
})

# -------------------------------------------------
# Optional verbose display of detailed statistics
# -------------------------------------------------
if VERBOSE:
    print("\nNormalized feature summary:")
    display(summary_df.round(6))

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nNormalization sanity check complete.")

